# $\color{cyan}{\text{Imports and Setup}}$

## $\color{yellow}{\text{Imports}}$

In [1]:
# Import required packages
import pandas as pd
import numpy as np
import pickle
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy.stats import linregress
from plotly.subplots import make_subplots

## $\color{yellow}{\text{Setup}}$

In [2]:
with open('dashboard_catalog.pkl', 'rb') as f:
    master_data_catalog = pickle.load(f)

print("Data Loaded")

Data Loaded


In [3]:
def create_interactive_dashboard(data_catalog):
    '''Creates an interactive explorer for comparing data sources and features.

    Args:
        data_catalog (dict): Nested mapping of analysis modes to named dataframes.
    '''

    # Initialises the state dictionary to track the last known values and prevent overwriting active filters
    state = {
        'updating': False,
        'last_mode': None,
        'last_ch': None,
        'last_x_src': None,
        'last_x_col': None,
        'last_y_src': None,
        'last_y_col': None,
        'excluded_chips': set(),
        'valid_chips': []
    }

    # Extracts a list of all available data sources from the absolute analysis mode
    all_sources = list(data_catalog['Absolute'].keys())

    # Initialises the mode toggle buttons for analysis selection
    mode_toggle = widgets.ToggleButtons(options=['Absolute', 'Stage Delta', 'Intra-stage Kinetics'], style={'description_width': 'initial'})
    
    # Initialises the checkbox for filtering Protein G stages in immobilisation sources
    prg_checkbox = widgets.Checkbox(value=False, description='PrG Stages Only (Immob)', tooltip='Filter Immobilisation metrics to PrG stages only', style={'description_width': 'initial'})
    
    # Groups the mode toggle and Protein G checkbox into a horizontal box
    top_controls = widgets.HBox([mode_toggle, prg_checkbox], layout=widgets.Layout(align_items='center', grid_gap='20px'))

    # Initialises the channel dropdown widget
    channel_dropdown = widgets.Dropdown(options=['Channel 1', 'Channel 2', 'Both'], value='Channel 1', description='Channel:')

    # Initialises the horizontal axis source dropdown widget
    x_source_drop = widgets.Dropdown(options=all_sources, value=all_sources[0], description='X Source:')

    # Initialises the vertical axis source dropdown widget
    y_source_drop = widgets.Dropdown(options=all_sources, value=all_sources[0], description='Y Source:')

    # Initialises the horizontal axis metric dropdown widget
    x_col_drop = widgets.Dropdown(description='X Metric:')

    # Initialises the vertical axis metric dropdown widget
    y_col_drop = widgets.Dropdown(description='Y Metric:')

    # Initialises the bounded float text input for the horizontal axis minimum filter
    x_min_input = widgets.BoundedFloatText(description='Min X:')

    # Initialises the bounded float text input for the horizontal axis maximum filter
    x_max_input = widgets.BoundedFloatText(description='Max X:')

    # Initialises the bounded float text input for the vertical axis minimum filter
    y_min_input = widgets.BoundedFloatText(description='Min Y:')

    # Initialises the bounded float text input for the vertical axis maximum filter
    y_max_input = widgets.BoundedFloatText(description='Max Y:')

    # Initialises the button to reset the horizontal axis filters to their original bounds
    x_reset_btn = widgets.Button(description='Reset X', button_style='info', tooltip='Reset X filters to original bounds')

    # Initialises the button to reset the vertical axis filters to their original bounds
    y_reset_btn = widgets.Button(description='Reset Y', button_style='info', tooltip='Reset Y filters to original bounds')

    # Initialises the integer slider for the horizontal axis histogram bins
    x_bins_slider = widgets.IntSlider(value=30, min=5, max=150, step=5, description='X Hist Bins:', layout=widgets.Layout(width='300px'))

    # Initialises the integer slider for the vertical axis histogram bins
    y_bins_slider = widgets.IntSlider(value=30, min=5, max=150, step=5, description='Y Hist Bins:', layout=widgets.Layout(width='300px'))

    def reset_x_filters(b):
        '''Resets the X-axis filters to their maximum available limits.'''

        # Assigns the absolute minimum allowed value to the horizontal minimum input
        x_min_input.value = x_min_input.min

        # Assigns the absolute maximum allowed value to the horizontal maximum input
        x_max_input.value = x_max_input.max

    def reset_y_filters(b):
        '''Resets the Y-axis filters to their maximum available limits.'''

        # Assigns the absolute minimum allowed value to the vertical minimum input
        y_min_input.value = y_min_input.min

        # Assigns the absolute maximum allowed value to the vertical maximum input
        y_max_input.value = y_max_input.max

    # Binds the horizontal reset function to the respective reset button click event
    x_reset_btn.on_click(reset_x_filters)

    # Binds the vertical reset function to the respective reset button click event
    y_reset_btn.on_click(reset_y_filters)

    # Groups the horizontal axis filters into a horizontal box container initially hidden from view
    x_filters = widgets.HBox([x_min_input, x_max_input, x_reset_btn], layout=widgets.Layout(display='none'))

    # Groups the vertical axis filters into a horizontal box container initially hidden from view
    y_filters = widgets.HBox([y_min_input, y_max_input, y_reset_btn], layout=widgets.Layout(display='none'))

    # Initialises the text input for dynamically filtering the available chips list
    chip_search = widgets.Text(placeholder='Filter available...', layout=widgets.Layout(width='95%'))
    
    # Initialises the selection box for displaying available chips
    chip_select = widgets.Select(options=[], rows=5, layout=widgets.Layout(width='95%'))
    
    # Groups the available chips search and selection widgets into the left vertical panel
    left_panel = widgets.VBox([widgets.HTML('<b>Available Chips:</b>'), chip_search, chip_select], layout=widgets.Layout(width='40%'))

    # Initialises the text input for dynamically filtering the excluded chips list
    excluded_search = widgets.Text(placeholder='Filter excluded...', layout=widgets.Layout(width='95%'))
    
    # Initialises the selection box for displaying excluded chips
    excluded_select = widgets.Select(options=[], rows=5, layout=widgets.Layout(width='95%'))
    
    # Groups the excluded chips search and selection widgets into the right vertical panel
    right_panel = widgets.VBox([widgets.HTML('<b>Excluded Chips:</b>'), excluded_search, excluded_select], layout=widgets.Layout(width='40%'))

    # Initialises the action button to exclude a single highlighted chip
    exclude_btn = widgets.Button(description='Exclude >', button_style='warning', layout=widgets.Layout(width='120px'))

    # Initialises the action button to exclude all currently valid chips
    exclude_all_btn = widgets.Button(description='Exclude All >>', button_style='danger', layout=widgets.Layout(width='120px'))

    # Initialises the action button to re-add a single highlighted chip to the available pool
    readd_btn = widgets.Button(description='< Re-add', button_style='info', layout=widgets.Layout(width='120px'))

    # Initialises the action button to clear all current exclusions
    reset_btn = widgets.Button(description='<< Reset All', button_style='success', layout=widgets.Layout(width='120px'))
    
    # Creates a blank HTML spacer to vertically align the action buttons with the adjacent text inputs
    button_spacer = widgets.HTML('<b>&nbsp;</b>')
    
    # Groups the spacer and all action buttons into the central vertical panel
    button_panel = widgets.VBox([button_spacer, exclude_btn, exclude_all_btn, readd_btn, reset_btn], layout=widgets.Layout(width='20%', justify_content='flex-start', align_items='center'))
    
    def refresh_ui(*args):
        '''Updates both chip listboxes based on current exclusions and dynamically valid chips.'''

        # Extracts and converts the available chips search term to lowercase
        avail_term = chip_search.value.lower()

        # Filters and updates the available chips selection list based on exclusions and the search term
        chip_select.options = [c for c in state['valid_chips'] if c not in state['excluded_chips'] and avail_term in c.lower()]
        
        # Extracts and converts the excluded chips search term to lowercase
        excl_term = excluded_search.value.lower()

        # Filters and updates the excluded chips selection list based on exclusions and the search term
        excluded_select.options = [c for c in state['valid_chips'] if c in state['excluded_chips'] and excl_term in c.lower()]

    # Binds the refresh function to the available chips search input changes
    chip_search.observe(refresh_ui, names='value')

    # Binds the refresh function to the excluded chips search input changes
    excluded_search.observe(refresh_ui, names='value')

    def exclude_selected(b):
        '''Excludes the currently highlighted chip from the available list.'''

        # Evaluates whether a valid chip is currently highlighted in the available list
        if chip_select.value:

            # Adds the highlighted chip to the excluded chips tracking set
            state['excluded_chips'].add(chip_select.value)

            # Refreshes the user interface to reflect the updated exclusion
            refresh_ui()

            # Triggers a redrawing of the plots to exclude the selected data
            plot_data()

    def exclude_all_filtered(b):
        '''Excludes all valid chips, regardless of the active search filter.'''
        
        # Evaluates whether there are any valid chips currently in the dataset
        if state['valid_chips']:
            
            # Adds all valid chips to the excluded chips tracking set
            state['excluded_chips'].update(state['valid_chips'])
            
            # Refreshes the user interface to reflect the complete exclusion
            refresh_ui()

            # Triggers a redrawing of the plots to exclude all data
            plot_data()

    def readd_selected(b):
        '''Restores the currently highlighted chip from the excluded list back to the active pool.'''

        # Evaluates whether a valid chip is currently highlighted in the excluded list and exists in the tracking set
        if excluded_select.value and excluded_select.value in state['excluded_chips']:

            # Removes the highlighted chip from the excluded chips tracking set
            state['excluded_chips'].remove(excluded_select.value)

            # Refreshes the user interface to reflect the restoration
            refresh_ui()

            # Triggers a redrawing of the plots to include the restored data
            plot_data()

    def reset_exclusions(b):
        '''Clears all current exclusions and restores the default chip lists.'''

        # Evaluates whether any chips are currently excluded
        if state['excluded_chips']:

            # Clears the excluded chips tracking set
            state['excluded_chips'].clear()

            # Resets the available chips search input
            chip_search.value = ''

            # Resets the excluded chips search input
            excluded_search.value = ''

            # Refreshes the user interface to display default lists
            refresh_ui()

            # Triggers a redrawing of the plots to include all data
            plot_data()

    # Binds the exclusion function to the single exclude button click event
    exclude_btn.on_click(exclude_selected)

    # Binds the batch exclusion function to the exclude all button click event
    exclude_all_btn.on_click(exclude_all_filtered)

    # Binds the restoration function to the re-add button click event
    readd_btn.on_click(readd_selected)

    # Binds the reset function to the reset exclusions button click event
    reset_btn.on_click(reset_exclusions)

    # Groups the left, middle, and right panels into the final horizontal exclusion control box
    exclusion_controls = widgets.HBox([left_panel, button_panel, right_panel], layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='10px 0px', width='100%'))
    
    # Initialises the output widget for the primary scatter plot
    out_scatter = widgets.Output()

    # Initialises the output widget for the secondary distribution plots
    out_dist = widgets.Output()

    def update_sources(*args):
        '''Updates available data sources depending on the selected channel.'''

        # Evaluates whether an update operation is already in progress
        if state['updating']: 

            # Returns control to prevent recursive updates
            return 

        # Flags the state to indicate an active update operation
        state['updating'] = True

        # Extracts the currently selected channel value
        ch = channel_dropdown.value

        # Determines the valid sources based on whether the secondary channel is explicitly selected
        valid_sources = [s for s in all_sources if s != 'Standard Curve'] if ch == 'Channel 2' else all_sources
        
        # Extracts the current horizontal axis source value
        curr_x = x_source_drop.value

        # Extracts the current vertical axis source value
        curr_y = y_source_drop.value

        # Updates the available options for the horizontal axis source dropdown
        x_source_drop.options = valid_sources

        # Updates the available options for the vertical axis source dropdown
        y_source_drop.options = valid_sources

        # Assigns the previous horizontal source if valid, otherwise falls back to the first available source
        x_source_drop.value = curr_x if curr_x in valid_sources else valid_sources[0]

        # Assigns the previous vertical source if valid, otherwise falls back to the first available source
        y_source_drop.value = curr_y if curr_y in valid_sources else valid_sources[0]

        # Flags the state to indicate the completion of the update operation
        state['updating'] = False

        # Triggers a cascading update of the dependent dropdown widgets
        update_dropdowns()

    def update_dropdowns(*args):
        '''Refreshes feature dropdown options after the selected sources change, applying the PrG filter if active.'''

        # Evaluates whether an update operation is already in progress
        if state['updating']: 

            # Returns control to prevent recursive updates
            return 

        # Flags the state to indicate an active update operation
        state['updating'] = True

        # Extracts the currently selected analysis mode
        mode = mode_toggle.value

        # Extracts the currently selected channel value
        ch = channel_dropdown.value

        # Assigns a fallback channel for extracting column names if both channels are selected
        col_ch = 'Channel 1' if ch == 'Both' else ch
        
        # Extracts the currently selected horizontal axis source
        x_src = x_source_drop.value

        # Extracts the currently selected vertical axis source
        y_src = y_source_drop.value

        # Extracts the current state of the Protein G filter checkbox
        prg_only = prg_checkbox.value

        # Evaluates whether both horizontal and vertical sources are selected
        if x_src and y_src:

            # Extracts the available column names for the selected horizontal source
            x_cols = list(data_catalog[mode][x_src][col_ch].columns)

            # Extracts the available column names for the selected vertical source
            y_cols = list(data_catalog[mode][y_src][col_ch].columns)

            # Evaluates whether the Protein G filter is currently active
            if prg_only:

                # Evaluates whether the horizontal source relates to immobilisation data
                if 'Immob' in x_src:

                    # Filters the horizontal column names to include only Protein G stages
                    x_cols = [col for col in x_cols if str(col).startswith('PrG')]

                # Evaluates whether the vertical source relates to immobilisation data
                if 'Immob' in y_src:

                    # Filters the vertical column names to include only Protein G stages
                    y_cols = [col for col in y_cols if str(col).startswith('PrG')]

            # Extracts the current horizontal axis metric value
            curr_x_col = x_col_drop.value

            # Extracts the current vertical axis metric value
            curr_y_col = y_col_drop.value

            # Updates the available options for the horizontal axis metric dropdown
            x_col_drop.options = x_cols

            # Updates the available options for the vertical axis metric dropdown
            y_col_drop.options = y_cols

            # Assigns the previous horizontal metric if valid, otherwise falls back to the first available metric
            x_col_drop.value = curr_x_col if curr_x_col in x_cols else (x_cols[0] if x_cols else None)

            # Assigns the previous vertical metric if valid, otherwise falls back to the first available metric
            y_col_drop.value = curr_y_col if curr_y_col in y_cols else (y_cols[0] if y_cols else None)

        # Flags the state to indicate the completion of the update operation
        state['updating'] = False

        # Triggers a cascading update of the dynamic filter bounds
        update_filter_bounds()

    def update_filter_bounds(*args):
        '''Dynamically sets the absolute min/max limits ONLY for the axes that were just modified.'''

        # Evaluates whether an update operation is already in progress
        if state['updating']: 

            # Returns control to prevent recursive updates
            return 

        # Flags the state to indicate an active update operation
        state['updating'] = True

        # Extracts the currently selected analysis mode and channel
        mode, ch = mode_toggle.value, channel_dropdown.value

        # Assigns a fallback channel for extracting limits if both channels are selected
        col_ch = 'Channel 1' if ch == 'Both' else ch
        
        # Extracts the currently selected sources for both axes
        x_src, y_src = x_source_drop.value, y_source_drop.value

        # Extracts the currently selected metrics for both axes
        x_col, y_col = x_col_drop.value, y_col_drop.value

        # Determines whether the horizontal axis parameters have changed since the last update
        needs_x_update = (mode != state.get('last_mode') or ch != state.get('last_ch') or x_src != state.get('last_x_src') or x_col != state.get('last_x_col'))

        # Determines whether the vertical axis parameters have changed since the last update
        needs_y_update = (mode != state.get('last_mode') or ch != state.get('last_ch') or y_src != state.get('last_y_src') or y_col != state.get('last_y_col'))

        # Evaluates whether any relevant changes occurred subsequent to the initial load
        if (needs_x_update or needs_y_update) and state.get('last_mode') is not None:

            # Clears the excluded chips tracking set
            state['excluded_chips'].clear()

            # Resets the available chips search input
            chip_search.value = ''

            # Resets the excluded chips search input
            excluded_search.value = ''

        # Defines a large numerical constant to temporarily remove boundary limits
        LARGE_NUM = 1e30 

        # Evaluates whether the horizontal axis filters require updating
        if needs_x_update:

            # Displays the horizontal axis filter controls
            x_filters.layout.display = 'flex'

            # Checks whether a valid horizontal metric is selected
            if x_col:

                # Retrieves the target dataframe from the data catalog
                df_x = data_catalog[mode][x_src][col_ch]

                # Evaluates whether the target dataframe and metric column are valid
                if not df_x.empty and x_col in df_x.columns:

                    # Extracts the target horizontal data and drops any missing values
                    x_data = df_x[x_col].dropna()

                    # Evaluates whether the extracted horizontal data is populated
                    if not x_data.empty:

                        # Calculates the absolute minimum and maximum boundaries for the horizontal data
                        x_min, x_max = float(x_data.min()), float(x_data.max())

                        # Adjusts the maximum boundary slightly if it equals the minimum to prevent widget errors
                        if x_min == x_max: 
                            x_max += 1e-9

                        # Temporarily expands the minimum input bounds to accept the new values
                        x_min_input.min, x_max_input.max = -LARGE_NUM, LARGE_NUM

                        # Temporarily expands the maximum input bounds to accept the new values
                        x_max_input.min, x_min_input.max = -LARGE_NUM, LARGE_NUM
                        
                        # Assigns the calculated boundaries as the current input values
                        x_min_input.value, x_max_input.value = x_min, x_max
                        
                        # Restricts the minimum input bounds to the newly calculated limits
                        x_min_input.min, x_min_input.max = x_min, x_max

                        # Restricts the maximum input bounds to the newly calculated limits
                        x_max_input.min, x_max_input.max = x_min, x_max
                        
            # Records the updated horizontal source and metric into the state tracking dictionary
            state['last_x_src'], state['last_x_col'] = x_src, x_col

        # Evaluates whether the vertical axis filters require updating
        if needs_y_update:

            # Displays the vertical axis filter controls
            y_filters.layout.display = 'flex'

            # Checks whether a valid vertical metric is selected
            if y_col:

                # Retrieves the target dataframe from the data catalog
                df_y = data_catalog[mode][y_src][col_ch]

                # Evaluates whether the target dataframe and metric column are valid
                if not df_y.empty and y_col in df_y.columns:

                    # Extracts the target vertical data and drops any missing values
                    y_data = df_y[y_col].dropna()

                    # Evaluates whether the extracted vertical data is populated
                    if not y_data.empty:

                        # Calculates the absolute minimum and maximum boundaries for the vertical data
                        y_min, y_max = float(y_data.min()), float(y_data.max())

                        # Adjusts the maximum boundary slightly if it equals the minimum to prevent widget errors
                        if y_min == y_max: 
                            y_max += 1e-9

                        # Temporarily expands the minimum input bounds to accept the new values
                        y_min_input.min, y_max_input.max = -LARGE_NUM, LARGE_NUM

                        # Temporarily expands the maximum input bounds to accept the new values
                        y_max_input.min, y_min_input.max = -LARGE_NUM, LARGE_NUM
                        
                        # Assigns the calculated boundaries as the current input values
                        y_min_input.value, y_max_input.value = y_min, y_max
                        
                        # Restricts the minimum input bounds to the newly calculated limits
                        y_min_input.min, y_min_input.max = y_min, y_max

                        # Restricts the maximum input bounds to the newly calculated limits
                        y_max_input.min, y_max_input.max = y_min, y_max
                        
            # Records the updated vertical source and metric into the state tracking dictionary
            state['last_y_src'], state['last_y_col'] = y_src, y_col

        # Records the updated mode and channel into the state tracking dictionary
        state['last_mode'], state['last_ch'] = mode, ch

        # Flags the state to indicate the completion of the update operation
        state['updating'] = False
        
        # Triggers a redrawing of the plots reflecting the updated boundaries
        plot_data()

    def get_joined_data(mode, x_src, y_src, x_col, y_col, channel_label):
        '''Joins selected dashboard fields and applies an optional channel filter.'''

        # Retrieves a copy of the horizontal dataframe from the data catalog
        df_x = data_catalog[mode][x_src][channel_label].copy()

        # Retrieves a copy of the vertical dataframe from the data catalog
        df_y = data_catalog[mode][y_src][channel_label].copy()

        # Evaluates whether either extracted dataframe is empty
        if df_x.empty or df_y.empty:

            # Returns an empty dataframe indicating a joining failure
            return pd.DataFrame()

        # Casts the horizontal dataframe column names to strings to ensure consistent merging
        df_x.columns = df_x.columns.astype(str)

        # Casts the vertical dataframe column names to strings to ensure consistent merging
        df_y.columns = df_y.columns.astype(str)

        # Casts the targeted horizontal and vertical metric column names to strings
        x_col_str, y_col_str = str(x_col), str(y_col)

        # Merges the targeted columns from both dataframes into a single structure using an inner join
        df_merged = pd.merge(df_x[[x_col_str]], df_y[[y_col_str]], left_index=True, right_index=True, how='inner', suffixes=('_x', '_y'))
        
        # Identifies the actual horizontal column name depending on potential suffixing during the merge
        actual_x_col = x_col_str + '_x' if x_col_str == y_col_str else x_col_str

        # Identifies the actual vertical column name depending on potential suffixing during the merge
        actual_y_col = y_col_str + '_y' if x_col_str == y_col_str else y_col_str
        
        # Renames the merged columns to standard generic identifiers
        df_merged = df_merged.rename(columns={actual_x_col: 'x_val', actual_y_col: 'y_val'})

        # Replaces infinite values with missing representations and drops any incomplete rows
        return df_merged.replace([np.inf, -np.inf], np.nan).dropna()

    def plot_data(*args):
        '''Calculates data for and constructs both the scatter plot and distribution subplots.'''
        
        # Evaluates whether an update operation is already in progress
        if state['updating']: 

            # Returns control to prevent recursive drawing
            return

        # Extracts the currently selected analysis mode and channel
        mode, channel = mode_toggle.value, channel_dropdown.value

        # Extracts the currently selected sources for both axes
        x_src, y_src = x_source_drop.value, y_source_drop.value

        # Extracts the currently selected metrics for both axes
        x_col, y_col = x_col_drop.value, y_col_drop.value

        # Evaluates whether valid metrics are selected for both axes
        if not x_col or not y_col:

            # Clears the valid chips tracking list
            state['valid_chips'] = []

            # Refreshes the user interface to reflect the empty chip list
            refresh_ui()

            # Enters the context of the primary scatter plot widget
            with out_scatter:

                # Clears the existing scatter plot output
                out_scatter.clear_output(wait=True)

                # Prints an instructional message prompting valid metric selection
                print('Please select valid metrics to plot.')

            # Enters the context of the secondary distribution plots widget
            with out_dist:

                # Clears the existing distribution plots output
                out_dist.clear_output(wait=True)

            # Returns control to halt further plotting execution
            return

        # Determines the specific channels to plot based on the dropdown selection
        channels_to_plot = ['Channel 1', 'Channel 2'] if channel == 'Both' else [channel]

        # Defines a dictionary mapping distinct colours to each channel
        colors = {'Channel 1': '#1f77b4', 'Channel 2': '#ff1e0e'}
        
        # Initialises an empty set to track valid chips for the current selection
        current_valid_chips = set()

        # Initialises an empty dictionary to temporarily store raw plotted dataframes
        raw_dfs = {}

        # Initialises an empty dictionary to store the final filtered dataframes
        filtered_dfs = {}

        # Loops through each specific channel targeted for plotting
        for ch in channels_to_plot:

            # Wraps the data extraction process in a try block to handle potential missing sources
            try:

                # Triggers the joining function to retrieve the combined data for the current channel
                df = get_joined_data(mode, x_src, y_src, x_col, y_col, ch)

                # Checks whether the extracted joined dataframe contains data
                if not df.empty:

                    # Maps the extracted dataframe to its corresponding channel in the raw data dictionary
                    raw_dfs[ch] = df

                    # Appends the chip identifiers present in the dataframe to the valid chips tracking set
                    current_valid_chips.update(df.index.astype(str).tolist())

            # Catches exceptions thrown during data retrieval without breaking the loop
            except Exception as e:

                # Proceeds to the next iteration
                continue
        
        # Updates the state tracking list with the sorted valid chips
        state['valid_chips'] = sorted(list(current_valid_chips))

        # Cleans the excluded chips tracking set to remove identifiers no longer relevant to the selection
        state['excluded_chips'] = {c for c in state['excluded_chips'] if c in state['valid_chips']}

        # Refreshes the user interface to update the chip selection lists
        refresh_ui()

        # Initialises a boolean flag to track if any data is successfully plotted
        plotted_any = False

        # Loops through each channel targeted for plotting to apply active filters
        for ch in channels_to_plot:

            # Evaluates whether raw data was successfully compiled for the current channel
            if ch not in raw_dfs: 
                
                # Skips to the next channel iteration
                continue
            
            # Extracts the compiled raw dataframe for the current channel
            plot_df = raw_dfs[ch]

            # Initialises a boolean mask populated with true values corresponding to the dataframe index
            mask = pd.Series(True, index=plot_df.index)

            # Updates the mask to filter horizontal values within the specified boundaries
            mask &= (plot_df['x_val'] >= x_min_input.value) & (plot_df['x_val'] <= x_max_input.value)

            # Updates the mask to filter vertical values within the specified boundaries
            mask &= (plot_df['y_val'] >= y_min_input.value) & (plot_df['y_val'] <= y_max_input.value)

            # Evaluates whether any chips are currently selected for exclusion
            if state['excluded_chips']:

                # Updates the mask to remove rows corresponding to the excluded chip identifiers
                mask &= ~plot_df.index.astype(str).isin(state['excluded_chips'])

            # Applies the aggregated boolean mask to filter the plot dataframe
            plot_df = plot_df[mask]

            # Evaluates whether the filtered dataframe retains sufficient data points for visualisation
            if len(plot_df) >= 2: 

                # Maps the filtered dataframe to its corresponding channel in the output dictionary
                filtered_dfs[ch] = plot_df

                # Sets the plotting flag to true indicating valid data is present
                plotted_any = True

        # Checks whether the active filters removed all available data records
        if not plotted_any:

            # Enters the context of the primary scatter plot widget
            with out_scatter:

                # Clears the existing scatter plot output
                out_scatter.clear_output(wait=True)

                # Prints an instructional message indicating insufficient data
                print('Not enough matching chip records to plot these selections within the specified filter bounds.')

            # Enters the context of the secondary distribution plots widget
            with out_dist:

                # Clears the existing distribution plots output
                out_dist.clear_output(wait=True)

            # Returns control to halt further plotting execution
            return

        # Initialises an empty Plotly figure object for the primary scatter plot
        fig = go.Figure()

        # Loops through each filtered dataframe prepared for plotting
        for ch, plot_df in filtered_dfs.items():

            # Extracts the targeted horizontal and vertical arrays from the filtered dataframe
            x_data, y_data = plot_df['x_val'], plot_df['y_val']

            # Adds a scatter trace plotting the extracted data points onto the figure
            fig.add_trace(go.Scatter(
                x=x_data, y=y_data, mode='markers', name=f'{ch}', 
                marker=dict(size=8, opacity=0.7, color=colors[ch], line=dict(width=1, color='DarkSlateGrey')), 
                text=plot_df.index, hovertemplate='Chip ID: %{text}<br>X: %{x:.4f}<br>Y: %{y:.4f}<extra></extra>'
            ))

            # Evaluates whether the horizontal data contains sufficient variance to calculate a linear fit
            if x_data.nunique() > 1:

                # Calculates the linear regression parameters spanning the scatter points
                slope, intercept, r_value, p_value, std_err = linregress(x_data, y_data)

                # Generates a dense array of linearly spaced horizontal coordinates bounding the data
                x_fit = np.linspace(x_data.min(), x_data.max(), 100)

                # Calculates the predicted vertical coordinates using the derived linear model
                y_fit = slope * x_fit + intercept

                # Adds a dashed line trace representing the linear fit onto the figure
                fig.add_trace(go.Scatter(
                    x=x_fit, y=y_fit, mode='lines', 
                    name=f'{ch} Fit (r={r_value:.3f}, R^2={r_value**2:.3f})', 
                    line=dict(color=colors[ch], dash='dash', width=2), hoverinfo='skip'
                ))

        # Constructs the title string incorporating the selected sources and metrics
        title = f'{y_src} [{y_col}] vs {x_src} [{x_col}]'

        # Updates the comprehensive layout and styling parameters of the scatter figure
        fig.update_layout(title=title, xaxis_title=f'{x_src} : {x_col}', yaxis_title=f'{y_src} : {y_col}', template='plotly_white', height=600, margin=dict(l=40, r=40, t=60, b=40), hovermode='closest')
        
        # Initialises a complex Plotly subplots figure for the secondary distributions
        dist_fig = make_subplots(
            rows=2, cols=2, 
            row_heights=[0.2, 0.8], 
            shared_xaxes=True,
            vertical_spacing=0.02,
            subplot_titles=(f'{x_col} Distribution', f'{y_col} Distribution', '', '')
        )

        # Extracts the histogram bin counts designated by the slider widgets
        x_bins = x_bins_slider.value
        y_bins = y_bins_slider.value

        # Loops through each filtered dataframe prepared for plotting
        for ch, df in filtered_dfs.items():

            # Adds a horizontal box plot for the primary axis distribution into the first subplot row
            dist_fig.add_trace(go.Box(x=df['x_val'], name=ch, marker_color=colors[ch], showlegend=False), row=1, col=1)

            # Adds a vertical box plot for the secondary axis distribution into the first subplot row
            dist_fig.add_trace(go.Box(x=df['y_val'], name=ch, marker_color=colors[ch], showlegend=False), row=1, col=2)
            
            # Adds a horizontal histogram representing the primary axis density into the second subplot row
            dist_fig.add_trace(go.Histogram(x=df['x_val'], nbinsx=x_bins, name=ch, marker_color=colors[ch], opacity=0.7, showlegend=False), row=2, col=1)

            # Adds a vertical histogram representing the secondary axis density into the second subplot row
            dist_fig.add_trace(go.Histogram(x=df['y_val'], nbinsx=y_bins, name=ch, marker_color=colors[ch], opacity=0.7, showlegend=False), row=2, col=2)

        # Updates the comprehensive layout and styling parameters of the distribution figure
        dist_fig.update_layout(barmode='overlay', template='plotly_white', height=400, margin=dict(l=40, r=40, t=40, b=40))

        # Enters the context of the primary scatter plot widget
        with out_scatter:

            # Clears the existing scatter plot output
            out_scatter.clear_output(wait=True)

            # Renders the newly generated scatter figure
            fig.show()

        # Enters the context of the secondary distribution plots widget
        with out_dist:

            # Clears the existing distribution plots output
            out_dist.clear_output(wait=True)

            # Renders the newly generated distribution figure
            dist_fig.show()

    # Binds the source update function to the channel dropdown value changes
    channel_dropdown.observe(update_sources, 'value')

    # Binds the dropdown update function to the analysis mode toggle changes
    mode_toggle.observe(update_dropdowns, 'value')

    # Binds the dropdown update function to the Protein G filter checkbox changes
    prg_checkbox.observe(update_dropdowns, 'value')

    # Binds the dropdown update function to the horizontal axis source dropdown changes
    x_source_drop.observe(update_dropdowns, 'value')

    # Binds the dropdown update function to the vertical axis source dropdown changes
    y_source_drop.observe(update_dropdowns, 'value')
    
    # Binds the boundary update function to the horizontal axis metric dropdown changes
    x_col_drop.observe(update_filter_bounds, 'value')

    # Binds the boundary update function to the vertical axis metric dropdown changes
    y_col_drop.observe(update_filter_bounds, 'value')
    
    # Binds the plot redrawing function to the horizontal minimum boundary changes
    x_min_input.observe(plot_data, 'value')

    # Binds the plot redrawing function to the horizontal maximum boundary changes
    x_max_input.observe(plot_data, 'value')

    # Binds the plot redrawing function to the vertical minimum boundary changes
    y_min_input.observe(plot_data, 'value')

    # Binds the plot redrawing function to the vertical maximum boundary changes
    y_max_input.observe(plot_data, 'value')
    
    # Binds the plot redrawing function to the horizontal and vertical histogram bins slider changes
    x_bins_slider.observe(plot_data, 'value')
    y_bins_slider.observe(plot_data, 'value')

    # Groups all upper interface control elements into a master vertical container
    controls = widgets.VBox([
        top_controls, 
        channel_dropdown, 
        widgets.HBox([x_source_drop, x_col_drop]), 
        x_filters, 
        widgets.HBox([y_source_drop, y_col_drop]), 
        y_filters, 
        exclusion_controls
    ])
    
    # Wraps the histogram bins sliders within an aligned horizontal box for layout styling
    bins_container = widgets.HBox([x_bins_slider, y_bins_slider], layout=widgets.Layout(padding='10px', justify_content='space-around'))

    # Displays the dashboard title header with defined styling
    display(widgets.HTML(f"<h3 style='margin-bottom:0px; color:#0000FF;'>Master Chip Comparison Dashboard</h3>"))

    # Renders the sequential layout of control containers and plot outputs
    display(controls, out_scatter, bins_container, out_dist)
    
    # Triggers the initial cascading source update to populate dropdowns and render the first plot
    update_sources()

# $\color{cyan}{\text{Dashboard Display}}$

In [4]:
# Launches the interactive cross-stage data exploration dashboard
create_interactive_dashboard(master_data_catalog)

HTML(value="<h3 style='margin-bottom:0px; color:#0000FF;'>Master Chip Comparison Dashboard</h3>")

Output()

Output()